In [98]:
import pandas as pd
import numpy as np

#### Load the dataset 

In [135]:
data=pd.read_csv("car_price_prediction.csv")
data.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [136]:
data.describe()
data.isnull().sum()
data.shape

(19237, 18)

In [137]:

data["Levy"]=data["Levy"].replace('-',np.nan)
data["Levy"]=pd.to_numeric(data["Levy"])
data["Levy"]=data['Levy'].fillna(data["Levy"].median())

data['Mileage']=data['Mileage'].str.replace("km","").astype(int)
data['Mileage'].dtype
data.head()
# data.isnull().sum()

data['Engine volume'] = data['Engine volume'].str.replace('Turbo','')
data['Engine volume'] = data['Engine volume'].astype(float)

data['Car_Age'] = 2024 - data['Prod. year']
data.drop(columns=['Prod. year'], inplace=True)

data.drop(columns=['Color'], inplace=True)


# data.head()
data.drop(columns=['Model'], inplace=True)

data=pd.get_dummies(data,drop_first=True,dtype=int)

data.drop(columns=["ID"], inplace = True)

data.head()

# data.select_dtypes(include="object").columns


,Price,Levy,Engine volume,Mileage,Cylinders,Airbags,Car_Age,Manufacturer_ALFA ROMEO,Manufacturer_ASTON MARTIN,Manufacturer_AUDI,...,Fuel type_Petrol,Fuel type_Plug-in Hybrid,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator,Drive wheels_Front,Drive wheels_Rear,Doors_04-May,Doors_>5,Wheel_Right-hand drive
0,13328,1399.0,3.5,186005,6.0,12,14,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,16621,1018.0,3.0,192000,6.0,8,13,0,0,0,...,1,0,0,1,0,0,0,1,0,0
2,8467,781.0,1.3,200000,4.0,2,18,0,0,0,...,1,0,0,0,1,1,0,1,0,1
3,3607,862.0,2.5,168966,4.0,0,13,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,11726,446.0,1.3,91901,4.0,4,10,0,0,0,...,1,0,0,0,0,1,0,1,0,0


In [138]:
data= data[data['Price'] < 100000]
data["Price"].describe()
data= data[data['Mileage']<500000]
data['Mileage'].describe()


count     18863.000000
mean     135623.001166
std       90166.311874
min           0.000000
25%       70000.000000
50%      125000.000000
75%      186000.000000
max      498815.000000
Name: Mileage, dtype: float64

In [139]:
data.to_csv("car_price_encoding.csv",index=False)

In [140]:
X=data.drop(columns=["Price"])
X
y=data["Price"]
y


0        13328
1        16621
2         8467
3         3607
4        11726
         ...  
19232     8467
19233    15681
19234    26108
19235     5331
19236      470
Name: Price, Length: 18863, dtype: int64

In [141]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=45)

In [142]:
# from sklearn.preprocessing import StandardScaler
# sc=StandardScaler()
# X_train=sc.fit_transform(X_train)
# X_test=sc.transform(X_test)


In [143]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [144]:
y_pred=model.predict(X_test)
y_pred

array([23077.19670324, 13229.72914586, 14257.72659926, ...,
        1843.95825133,  5423.42924626,  4501.84862266], shape=(3773,))

In [145]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
print("R2 Score :",r2_score(y_test,y_pred))
print("MSE :",mean_squared_error(y_test,y_pred))
print("MAE :",mean_absolute_error(y_test,y_pred))


R2 Score : 0.34003478059230585
MSE : 162959073.45171192
MAE : 9059.993680564594


In [146]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

RFR=RandomForestRegressor(
    n_estimators=200,
    random_state=45
)
RFR.fit(X_train,y_train)
y_pred_RFR=RFR.predict(X_test)

print("RFR R2 Score :",r2_score(y_test,y_pred_RFR))
print("RFR MSE :",mean_squared_error(y_test,y_pred_RFR))
print("RFR MAE :",mean_absolute_error(y_test,y_pred_RFR))

RFR R2 Score : 0.7834862010246907
RFR MSE : 53461738.638579056
RFR MAE : 4020.6235299184227


In [147]:
DTR=DecisionTreeRegressor()
DTR.fit(X_train,y_train)
y_pred_DTR=DTR.predict(X_test)

print("GBR R2 Score :",r2_score(y_test,y_pred_DTR))
print("GBR MSE :",mean_squared_error(y_test,y_pred_DTR))
print("GBR MAE :",mean_absolute_error(y_test,y_pred_DTR))

GBR R2 Score : 0.6263936203774567
GBR MSE : 92251148.4977634
GBR MAE : 4862.750232159541


In [148]:
GBR=GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)
GBR.fit(X_train,y_train)
y_pred_GBR=GBR.predict(X_test)

print("GBR R2 Score :",r2_score(y_test,y_pred_GBR))
print("GBR MSE :",mean_squared_error(y_test,y_pred_GBR))
print("GBR MAE :",mean_absolute_error(y_test,y_pred_GBR))

GBR R2 Score : 0.725104811297585
GBR MSE : 67877312.21273023
GBR MAE : 5278.752609552145


In [149]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
     n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("XGB R2 Score :", r2_score(y_test, y_pred))
print("XGB MSE :", mean_squared_error(y_test, y_pred))
print("XGB MAE :", mean_absolute_error(y_test, y_pred))

XGB R2 Score : 0.7946334481239319
XGB MSE : 50709256.0
XGB MAE : 4142.4072265625


In [150]:
results = {
    "Linear Regression":0.3400 ,
    "Decision Tree": 0.6263,
    "Random Forest": 0.7834,
    "Gradient Boosting": 0.7251,
    "XGB ":0.7946
}

pd.DataFrame(results.items(), columns=["Model","R2 Score"])

,Model,R2 Score
0,Linear Regression,0.3400
1,Decision Tree,0.6263
2,Random Forest,0.7834
3,Gradient Boosting,0.7251
4,XGB,0.7946
